# Lightpanda vs Google Chrome — TodoMVC GitHub Actions Pipeline Report

## Use Case & Test Scenario

This notebook analyses GitHub Actions CI pipeline runs that execute three Playwright end-to-end tests against the [TodoMVC React demo](https://todomvc.com/examples/react/dist/) — a canonical to-do list web application built in React.

**The three tests verify core user interactions:**

| # | Test name | What it does |
|---|-----------|---------------|
| 1 | `add a todo item` | Types "Buy groceries" into the input field, presses Enter, asserts the item appears in the list |
| 2 | `complete a todo item` | Adds an item, clicks its checkbox toggle, asserts the item is marked completed |
| 3 | `filter completed todos` | Adds an item, marks it completed, clicks the "Completed" filter link, asserts exactly one completed item is visible |

Both engines connect via the **Chrome DevTools Protocol (CDP)** so the same Playwright test code runs unmodified against either browser:
- **Google Chrome** — full-featured Chromium browser (headless), the industry standard baseline
- **Lightpanda** — a lightweight, purpose-built headless browser written in Zig, designed to minimise resource usage in CI/cloud environments

**What the pipeline measures** (columns in `pipeline_metrics.csv`):
- `setup_overhead_sec` — GitHub Actions job steps before tests run: checkout, Node install, npm ci, browser install/start
- `execution_time_sec` — time from first test to last test completion (3 Playwright tests)
- `teardown_overhead_sec` — post-test cleanup steps
- `total_duration_sec` — full wall-clock job time (sum of all three)

**Research question:** Does Lightpanda offer a meaningful pipeline speedup over Chrome for a realistic, small-scale Playwright test suite in GitHub Actions?

## 1. Environment Setup

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
pd.options.display.float_format = '{:,.3f}'.format

colors = {'Lightpanda': '#1f77b4', 'Google Chrome': '#ff7f0e'}
expected_engines = ['Lightpanda', 'Google Chrome']

csv_path = Path('pipeline_metrics.csv')
print(f'CSV: {csv_path.resolve()}')

CSV: C:\Users\Andi\repos\CDAS_SS2026\cdas-lightpanda\scripts\01_TodoMVC_3_tests\pipeline_metrics.csv


## 2. Load & Prepare Data

In [3]:
df = pd.read_csv(csv_path, sep=';')
df.columns = [c.strip() for c in df.columns]

# Parse timestamps
for col in ['started_at', 'completed_at']:
    df[col] = pd.to_datetime(df[col], utc=True)

# Numeric cols
numeric_cols = ['total_duration_sec', 'setup_overhead_sec', 'execution_time_sec', 'teardown_overhead_sec']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Normalize engine labels
engine_map = {'lightpanda': 'Lightpanda', 'google chrome': 'Google Chrome', 'chrome': 'Google Chrome'}
df['engine'] = df['engine'].astype(str).str.strip().str.lower().map(engine_map).fillna(df['engine'])

# Sort chronologically within each engine, assign run index
df = df.sort_values(['engine', 'started_at']).reset_index(drop=True)
df['run_index'] = df.groupby('engine').cumcount() + 1

df[['engine', 'run_id', 'started_at', 'run_index'] + numeric_cols].head(6)

,engine,run_id,started_at,run_index,total_duration_sec,setup_overhead_sec,execution_time_sec,teardown_overhead_sec
0,Google Chrome,26235543090,2026-05-21 15:24:37+00:00,1,23,15,5,0
1,Google Chrome,26338284707,2026-05-23 16:51:05+00:00,2,31,25,3,1
2,Google Chrome,26338525973,2026-05-23 17:02:54+00:00,3,17,11,4,1
3,Google Chrome,26338570053,2026-05-23 17:05:07+00:00,4,21,13,5,0
4,Google Chrome,26338587272,2026-05-23 17:05:57+00:00,5,20,13,4,0
5,Google Chrome,26338602770,2026-05-23 17:06:44+00:00,6,25,18,4,1


## 3. Data Validation

In [ ]:
counts = df['engine'].value_counts()
missing = df[numeric_cols].isna().sum()
negative = (df[numeric_cols] < 0).sum()

print('Rows per engine:')
print(counts.to_string())
print('\nMissing values:')
print(missing.to_string())
print('\nNegative values:')
print(negative.to_string())

assert set(expected_engines).issubset(set(df['engine'].unique())), 'Engine mismatch'
assert missing.sum() == 0, 'Missing values found'
assert negative.sum() == 0, 'Negative timing values found'
print('\nValidation passed.')

## 4. Summary Statistics

In [ ]:
def p90(x): return np.percentile(x, 90)
def p10(x): return np.percentile(x, 10)

summary = (
    df.groupby('engine')[numeric_cols]
    .agg(['mean', 'median', 'std', 'min', 'max', p10, p90])
    .round(2)
)
summary.columns = [f'{m}_{s}' for m, s in summary.columns]
display(summary)

# Delta table: Chrome vs Lightpanda
lp = summary.loc['Lightpanda']
ch = summary.loc['Google Chrome']

delta_rows = []
for col in numeric_cols:
    ch_mean = ch[f'{col}_mean']
    lp_mean = lp[f'{col}_mean']
    ch_med  = ch[f'{col}_median']
    lp_med  = lp[f'{col}_median']
    delta_rows.append({
        'Metric': col,
        'LP mean (s)': lp_mean,
        'Chrome mean (s)': ch_mean,
        'Chrome Δ% vs LP (mean)': f"{(ch_mean - lp_mean) / lp_mean * 100:+.1f}%",
        'LP median (s)': lp_med,
        'Chrome median (s)': ch_med,
        'Chrome Δ% vs LP (median)': f"{(ch_med - lp_med) / lp_med * 100:+.1f}%",
    })

delta_df = pd.DataFrame(delta_rows).set_index('Metric')
display(delta_df)

## 5. Charts

### 5.1 Mean Pipeline Phase Breakdown (Stacked Bar)

In [ ]:
phase_cols = ['setup_overhead_sec', 'execution_time_sec', 'teardown_overhead_sec']
phase_labels = ['Setup overhead', 'Test execution', 'Teardown overhead']
phase_colors = ['#457b9d', '#e76f51', '#adb5bd']

means = df.groupby('engine')[phase_cols].mean()

fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(expected_engines))
x = np.arange(len(expected_engines))

for col, label, color in zip(phase_cols, phase_labels, phase_colors):
    vals = [means.loc[e, col] for e in expected_engines]
    bars = ax.bar(x, vals, bottom=bottom, label=label, color=color, width=0.5)
    for i, (b, v) in enumerate(zip(bottom, vals)):
        if v > 0.5:
            ax.text(i, b + v / 2, f'{v:.1f}s', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += np.array(vals)

# Annotate total
for i, e in enumerate(expected_engines):
    total = means.loc[e, phase_cols].sum()
    ax.text(i, total + 0.3, f'Total: {total:.1f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(expected_engines)
ax.set_ylabel('Mean duration (seconds)')
ax.set_title('Mean pipeline phase breakdown per engine (10 runs each)')
ax.legend(loc='upper right')
ax.set_ylim(0, bottom.max() * 1.2)
plt.tight_layout()
plt.show()

### 5.2 Mean Values Per Metric (Side-by-Side Bar)

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))

for ax, col in zip(axes, numeric_cols):
    vals = [df[df['engine'] == e][col].mean() for e in expected_engines]
    bars = ax.bar(expected_engines, vals, color=[colors[e] for e in expected_engines], width=0.5)
    ax.bar_label(bars, fmt='%.1fs', fontsize=9, padding=3)
    ax.set_title(col.replace('_', ' ').replace(' sec', ' (s)'), fontsize=10)
    ax.set_ylabel('Mean (seconds)')
    ax.set_ylim(0, max(vals) * 1.3)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Mean pipeline metrics per engine — all 10 runs', fontsize=12)
plt.tight_layout()
plt.show()

### 5.3 Per-Run Timeline (Total Duration)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for ax, col, title in zip(
    axes,
    ['total_duration_sec', 'execution_time_sec'],
    ['Total pipeline duration per run', 'Test execution time per run']
):
    for engine in expected_engines:
        d = df[df['engine'] == engine].sort_values('run_index')
        ax.plot(d['run_index'], d[col], marker='o', linewidth=1.8,
                label=engine, color=colors[engine])
        ax.axhline(d[col].mean(), linestyle='--', linewidth=1,
                   color=colors[engine], alpha=0.6,
                   label=f'{engine} mean ({d[col].mean():.1f}s)')
    ax.set_title(title)
    ax.set_ylabel('Seconds')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[-1].set_xlabel('Run index')
plt.tight_layout()
plt.show()

### 5.4 Distribution Boxplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ['total_duration_sec', 'execution_time_sec']):
    data = [df[df['engine'] == e][col].values for e in expected_engines]
    bp = ax.boxplot(data, labels=expected_engines, patch_artist=True, showfliers=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, engine in zip(bp['boxes'], expected_engines):
        patch.set_facecolor(colors[engine])
        patch.set_alpha(0.7)
    ax.set_title(f'Distribution: {col.replace("_", " ").replace(" sec", " (s)")}')
    ax.set_ylabel('Seconds')
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Spread and variability across 10 runs per engine', fontsize=12)
plt.tight_layout()
plt.show()

### 5.5 Paired Run Speedup (Chrome / Lightpanda)

Runs are paired by temporal order (1st LP run vs 1st Chrome run, etc.), since both pipelines triggered simultaneously per workflow run.

In [ ]:
lp_df = df[df['engine'] == 'Lightpanda'].sort_values('run_index').reset_index(drop=True)
ch_df = df[df['engine'] == 'Google Chrome'].sort_values('run_index').reset_index(drop=True)

n_pairs = min(len(lp_df), len(ch_df))
pivot = pd.DataFrame({
    'run_index': lp_df['run_index'][:n_pairs],
    'total_speedup': ch_df['total_duration_sec'][:n_pairs].values / lp_df['total_duration_sec'][:n_pairs].values,
    'exec_speedup':  ch_df['execution_time_sec'][:n_pairs].values  / lp_df['execution_time_sec'][:n_pairs].values,
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes,
    ['total_speedup', 'exec_speedup'],
    ['Total duration speedup (Chrome / Lightpanda)', 'Test execution speedup (Chrome / Lightpanda)']
):
    ax.bar(pivot['run_index'], pivot[col], color='#2a9d8f', alpha=0.85)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label='Parity (1.0×)')
    geo_mean = float(np.exp(np.log(pivot[col]).mean()))
    ax.axhline(geo_mean, color='darkred', linestyle=':', linewidth=1.5,
               label=f'Geo mean: {geo_mean:.2f}×')
    ax.set_title(title)
    ax.set_xlabel('Run index')
    ax.set_ylabel('Speedup ratio (>1 = LP faster)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Per-run speedup of Lightpanda over Google Chrome', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Total duration  — geo mean speedup: {float(np.exp(np.log(pivot["total_speedup"]).mean())):.3f}×')
print(f'Test execution  — geo mean speedup: {float(np.exp(np.log(pivot["exec_speedup"]).mean())):.3f}×')
print(f'LP faster on total duration in {int((pivot["total_speedup"] > 1).sum())}/{n_pairs} runs')
print(f'LP faster on test execution  in {int((pivot["exec_speedup"]  > 1).sum())}/{n_pairs} runs')

### 5.6 Setup Overhead vs Execution Time (Scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for engine in expected_engines:
    d = df[df['engine'] == engine]
    ax.scatter(d['setup_overhead_sec'], d['execution_time_sec'],
               color=colors[engine], label=engine, s=80, alpha=0.8, edgecolors='white', linewidths=0.5)

ax.set_xlabel('Setup overhead (seconds)')
ax.set_ylabel('Test execution time (seconds)')
ax.set_title('Setup overhead vs test execution time per run')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary & Findings

In [ ]:
lp_means = df[df['engine'] == 'Lightpanda'][numeric_cols].mean()
ch_means = df[df['engine'] == 'Google Chrome'][numeric_cols].mean()

total_geo   = float(np.exp(np.log(pivot['total_speedup']).mean()))
exec_geo    = float(np.exp(np.log(pivot['exec_speedup']).mean()))
lp_wins_tot = int((pivot['total_speedup'] > 1).sum())
lp_wins_exc = int((pivot['exec_speedup']  > 1).sum())

findings = pd.DataFrame([
    {'Metric': 'Total pipeline duration', 'LP mean (s)': lp_means['total_duration_sec'],
     'Chrome mean (s)': ch_means['total_duration_sec'],
     'Chrome Δ%': f"{(ch_means['total_duration_sec'] - lp_means['total_duration_sec']) / lp_means['total_duration_sec'] * 100:+.1f}%",
     'Note': f'LP {total_geo:.2f}× faster (geo mean)'},
    {'Metric': 'Test execution time', 'LP mean (s)': lp_means['execution_time_sec'],
     'Chrome mean (s)': ch_means['execution_time_sec'],
     'Chrome Δ%': f"{(ch_means['execution_time_sec'] - lp_means['execution_time_sec']) / lp_means['execution_time_sec'] * 100:+.1f}%",
     'Note': f'LP {exec_geo:.2f}× faster (geo mean)'},
    {'Metric': 'Setup overhead', 'LP mean (s)': lp_means['setup_overhead_sec'],
     'Chrome mean (s)': ch_means['setup_overhead_sec'],
     'Chrome Δ%': f"{(ch_means['setup_overhead_sec'] - lp_means['setup_overhead_sec']) / lp_means['setup_overhead_sec'] * 100:+.1f}%",
     'Note': 'LP binary install faster than Chrome/Chromium download'},
    {'Metric': 'Teardown overhead', 'LP mean (s)': lp_means['teardown_overhead_sec'],
     'Chrome mean (s)': ch_means['teardown_overhead_sec'],
     'Chrome Δ%': 'n/a' if lp_means['teardown_overhead_sec'] == 0 else f"{(ch_means['teardown_overhead_sec'] - lp_means['teardown_overhead_sec']) / lp_means['teardown_overhead_sec'] * 100:+.1f}%",
     'Note': 'Minimal for both'},
]).set_index('Metric')

display(findings)

print(f"""
Key Findings
============
Pipeline speed
  - Total job duration : Lightpanda {total_geo:.2f}× faster (geo mean over {n_pairs} paired runs)
  - Test execution only: Lightpanda {exec_geo:.2f}× faster (same 3 Playwright tests, both via CDP)
  - LP faster on total wall-clock in {lp_wins_tot}/{n_pairs} runs
  - LP faster on test execution  in {lp_wins_exc}/{n_pairs} runs

Overhead breakdown
  - Setup dominates total job time for both engines (~{lp_means['setup_overhead_sec']:.0f}s LP, ~{ch_means['setup_overhead_sec']:.0f}s Chrome)
  - LP setup faster because Lightpanda binary is smaller than full Chromium download
  - Test execution is short (~{lp_means['execution_time_sec']:.0f}s LP, ~{ch_means['execution_time_sec']:.0f}s Chrome) — 3 lightweight UI tests

Compatibility
  - All 3 TodoMVC tests pass on both engines — Lightpanda handles React rendering via CDP
  - Hash-based URL filtering (#/completed) works on Lightpanda for the 3 selected tests
""")